# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets, fields, and their `@id`s using the Croissant metadata structure.

In [ ]:
# List available record sets and their IDs
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the dataset.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- {rs['@id']} | name: {rs.get('name', 'N/A')} | description: {rs.get('description', 'N/A')}")

    # For exploration, look at the first available record set
    record_set_id = record_sets[0]['@id']
    print(f"\nExample fields in record set '{record_set_id}':")
    for field in record_sets[0]['field']:
        print(f"  - {field['@id']} | name: {field.get('name','N/A')} | dataType: {field.get('dataType','N/A')}")

    # Show some records from the record set
    print(f"\nSample records from record set '{record_set_id}':")
    for idx, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if idx >= 2:
            break

## 3. Data Extraction
Load records from each record set into a pandas DataFrame for further analysis. All entities are referenced by their `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

# Show columns of the main clinical data record set (using the first record set)
if record_set_ids:
    chosen_record_set = record_set_ids[0]
    print("\nColumns in record set:", chosen_record_set)
    print(dataframes[chosen_record_set].columns.tolist())
    dataframes[chosen_record_set].head()
else:
    print('No record sets available for data extraction.')

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic EDA steps: filter for patients above a certain age, normalize a numeric column (e.g., age), and group by a key attribute (e.g., sex).

In [ ]:
# We'll use the first record set and guess a numeric field, e.g., age
main_rs_id = record_set_ids[0] if record_set_ids else None

# Try to pick a numeric field from the record set fields
if main_rs_id is not None:
    main_fields = next(rs for rs in dataset.record_sets if rs['@id'] == main_rs_id)['field']

    # Look for a field likely representing 'age', fallback to another numeric field
    age_field_id = None
    other_numeric_field_id = None
    for field in main_fields:
        if field.get('dataType','').lower() in ['integer', 'float', 'number']:
            if 'age' in field.get('name','').lower() or 'age' in field['@id'].lower():
                age_field_id = field['@id']
                break
            if not other_numeric_field_id:
                other_numeric_field_id = field['@id']
    numeric_field_id = age_field_id or other_numeric_field_id
    if not numeric_field_id:
        print('No numeric field found for EDA.')

    else:
        df = dataframes[main_rs_id]
        # Check that field exists and has valid values
        if numeric_field_id in df.columns:
            print(f"Using numeric field: {numeric_field_id}")
            # Convert to numeric
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            # Apply a filter threshold (for illustration, e.g., age > 50)
            threshold = 50
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            print(filtered_df.head())

            # Normalize the field
            filtered_df[f"{numeric_field_id}_normalized"] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
                filtered_df[numeric_field_id].std()
            )
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Try grouping by sex or another likely categorical attribute
            group_field_id = None
            for field in main_fields:
                if 'sex' in field.get('name','').lower() or 'sex' in field['@id'].lower() or 'gender' in field.get('name','').lower():
                    group_field_id = field['@id']
                    break
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
                print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
                print(grouped_df.head())
        else:
            print(f"Chosen numeric field {numeric_field_id} not found in data columns.")
else:
    print('No record set to perform EDA.')

## 5. Visualization
Visualize data distributions of the selected numeric field (e.g., histogram of age for filtered patients) and relationships between fields (e.g., age by sex).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# For illustrative purposes, plot histogram and boxplot if previous cells succeeded
if main_rs_id is not None and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by sex (if available)
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook has demonstrated how to load and explore the FAIR² dataset using the `mlcroissant` library.

- We located the dataset's record sets using their `@id` fields and inspected their fields and types.
- Data was extracted as pandas DataFrames directly from the Croissant schema definitions.
- Simple EDA and visualizations were performed using representative fields (such as patient age), referenced by their `@id`s.

For more detailed analysis, consult the Croissant schema documentation or explore additional record sets and variables within the dataset.